In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

In [0]:
from pyspark.sql.functions import col, lit, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [0]:
movie_cast_df = (
    spark.read
        .format("json")
        .option("multiline", "true")
        .load(f"{bronze_folder_path}/movie_cast.json")
    )
display(movie_cast_df)

In [0]:
movie_cast_df.columns

In [0]:
movie_cast_schema = StructType([
    StructField("castOrder", IntegerType(), True),
    StructField("characterName", StringType(), True),
    StructField("genderId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("personId", IntegerType(), True)
])
movie_cast_df = (
    spark.read
        .schema(movie_cast_schema)
        .format("json")
        .option("multiline", "true")
        .load(f"{bronze_folder_path}/movie_cast.json")
)

In [0]:
movie_cast_df = (
    add_ingestion_date(movie_cast_df)
        .select(
            col("movieId").alias("movie_id"),
            col("personId").alias("person_id"),
            col("characterName").alias("character_name"),
            lit(v_env).alias("env")
        )
)
display(movie_cast_df)

In [0]:
(
    movie_cast_df
        .write
        .format("parquet")
        .mode("overwrite")
        .save(f"{silver_folder_path}/movie_cast")
)

In [0]:
dbutils.notebook.exit("Success")